In [3]:
import os
import re
import csv
from collections import defaultdict
from statistics import mean, stdev

def parse_log_file_last_metrics(path):
    eval_metrics = {}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    i = len(lines) - 1
    while i >= 0:
        if lines[i].strip().startswith("start to eval"):
            if i + 1 < len(lines) and lines[i+1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.eE+-]+)", lines[i+1])
                if m: eval_metrics["hit20"] = float(m.group(1))
            if i + 2 < len(lines) and lines[i+2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.eE+-]+)", lines[i+2])
                if m: eval_metrics["hit50"] = float(m.group(1))
            if i + 3 < len(lines) and lines[i+3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.eE+-]+)", lines[i+3])
                if m: eval_metrics["hit100"] = float(m.group(1))
            if i + 4 < len(lines) and lines[i+4].startswith("roc_auc"):
                fields = ['roc_auc', 'pr_auc', 'f1', 'mrr']
                nums = [float(x) for x in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', lines[i+4])][-4:]
                result_dict = dict(zip(fields, nums))
                for k, v in result_dict.items():
                    eval_metrics[k] = v
            if eval_metrics:
                break
        i -= 1
    return eval_metrics

# ==== 用户指定参数 ====
dataset = "cora"
pos_ratios = [1]
c = 1
hs = [14, 15]
gammas = [6, 7, 8]
alphas = [30, 40, 50, 60, 75, 90, 105, 120, 140, 160, 180, 200]
epoch = 100
seed = 1
ratio = 0.4
convergence = 0.8

log_base_dir = f"."
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

ALL_METRICS = ['hit20', 'hit50', 'hit100', 'roc_auc', 'pr_auc', 'f1', 'mrr']

# ==== 聚合结果 ====
metric_bucket = defaultdict(list)
metric_keys = set()


for pos_ratio in pos_ratios:
    for h in hs:
        for gamma in gammas:
            for alpha in alphas:
                fname = f"s{seed}-h{h}-c{c}-p{pos_ratio}-r{ratio}-g{gamma}-a{alpha}-e{epoch}-con{convergence}.log"
                log_path = os.path.join(log_base_dir, fname)

                key = (ratio, pos_ratio, c, h, gamma, alpha)

                if not os.path.isfile(log_path):
                    print(f"[WARN] 缺失: {log_path}")
                    # 也补 0（即使文件缺失）
                    metrics = {k: 0.0 for k in ALL_METRICS}
                    metric_bucket[key].append(metrics)
                    metric_keys.update(metrics.keys())
                    continue

                metrics = parse_log_file_last_metrics(log_path)

                if not metrics:
                    print(f"[WARN] 无 test 指标: {log_path}")
                    metrics = {k: 0.0 for k in ALL_METRICS}

                metric_bucket[key].append(metrics)
                metric_keys.update(metrics.keys())

# ==== 写入 CSV ====
csv_path = os.path.join(output_dir, f"{dataset}_grid_result.csv")
metric_keys = sorted(metric_keys)
header = ["split_ratio", "pos_ratio", "h", "gamma", "alpha"] + metric_keys

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    for key, metrics_list in metric_bucket.items():
        ratio, pos_ratio, c, h, gamma, alpha = key
        row = {
            "split_ratio": ratio,
            "pos_ratio": pos_ratio,
            "h": h,
            "gamma": gamma,
            "alpha": alpha
        }
        for k in metric_keys:
            vals = [m[k] for m in metrics_list if k in m]
            if vals:
                mu = round(round(mean(vals), 4) * 100, 2)
                row[k] = f"{mu}"
            else:
                row[k] = "0.0"
        writer.writerow(row)

print(f"[INFO] 写入完毕: {csv_path}")


[INFO] 写入完毕: ./results/cora_grid_result.csv


In [ ]:
import pandas as pd
import plotly.graph_objs as go
from ipywidgets import interact, SelectionSlider, Dropdown
import numpy as np

# ==== 读取数据 ====
dataset = "cora"
csv_path = f"./results/{dataset}_grid_result.csv"
df = pd.read_csv(csv_path)

# ==== 类型标准化 ====
PARAM_COLS = ['split_ratio', 'pos_ratio', 'h', 'gamma', 'alpha']
for col in ['split_ratio','pos_ratio','alpha']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in ['h','gamma']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# ==== 指标列自动识别 ====
metric_choices = [c for c in df.columns if c not in PARAM_COLS]

# ==== 解析单元格：同时支持 “数值” 或 “均值 ± 标准差/方差” ====
def parse_metric_cell(val):
    if pd.isna(val):
        return None, None
    # 纯数值
    if isinstance(val, (int, float, np.number)):
        return float(val), 0.0
    # 字符串情况
    s = str(val).strip()
    if not s:
        return None, None
    s = s.replace('+/-', '±')
    if '±' in s:
        left, right = s.split('±', 1)
        try:
            return float(left.strip()), float(right.strip())
        except Exception:
            return None, None
    # 仅有一个数字的字符串
    try:
        return float(s), 0.0
    except Exception:
        return None, None

# ==== 可视化函数 ====
def plot_metric(metric_name, split_ratio, pos_ratio, h, gamma):
    # 过滤：严格相等（离散选项避免精度问题）
    filtered = df[
        (df['split_ratio'] == split_ratio) &
        (df['pos_ratio'] == pos_ratio) &
        (df['h'] == h) &
        (df['gamma'] == gamma)
    ]

    if filtered.empty:
        print("无数据匹配该配置")
        return

    # 收集 alpha-指标
    alphas, mus, stds = [], [], []
    for _, row in filtered.iterrows():
        alpha = row['alpha']
        mu, sd = parse_metric_cell(row.get(metric_name))
        if mu is not None:
            alphas.append(float(alpha))
            mus.append(mu)
            stds.append(0.0 if sd is None else sd)

    if len(alphas) == 0:
        print("有匹配行，但该指标在这些行中均为空或不可解析。")
        return

    # 排序
    order = np.argsort(alphas)
    alphas = np.array(alphas)[order]
    mus    = np.array(mus)[order]
    stds   = np.array(stds)[order]

    # 绘图
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=alphas,
        y=mus,
        error_y=dict(type='data', array=stds, visible=bool(np.any(stds))),
        mode='lines+markers',
        name=metric_name
    ))
    fig.update_layout(
        title=f"{metric_name} vs Alpha",
        xaxis_title="alpha",
        yaxis_title=metric_name,
        template="simple_white"
    )
    fig.show()

# ==== 交互控件：使用离散选项滑块（SelectionSlider）避免精度问题 ====
split_ratio_slider = SelectionSlider(
    options=sorted(df['split_ratio'].dropna().unique().tolist()),
    value=sorted(df['split_ratio'].dropna().unique().tolist())[0],
    description='split_ratio'
)
pos_ratio_slider = SelectionSlider(
    options=sorted(df['pos_ratio'].dropna().unique().tolist()),
    value=sorted(df['pos_ratio'].dropna().unique().tolist())[0],
    description='pos_ratio'
)
h_slider = SelectionSlider(
    options=sorted([int(x) for x in df['h'].dropna().unique().tolist()]),
    value=sorted([int(x) for x in df['h'].dropna().unique().tolist()])[0],
    description='h'
)
gamma_slider = SelectionSlider(
    options=sorted([int(x) for x in df['gamma'].dropna().unique().tolist()]),
    value=sorted([int(x) for x in df['gamma'].dropna().unique().tolist()])[0],
    description='gamma'
)

interact(
    plot_metric,
    metric_name=Dropdown(options=metric_choices, description='指标:'),
    split_ratio=split_ratio_slider,
    pos_ratio=pos_ratio_slider,
    h=h_slider,
    gamma=gamma_slider
)


interactive(children=(Dropdown(description='指标:', options=('f1', 'hit100', 'hit20', 'hit50', 'mrr', 'pr_auc', …

<function __main__.plot_metric(metric_name, split_ratio, pos_ratio, h, gamma)>

In [3]:
import os
import pandas as pd
import numpy as np

# 可选：尝试更强的统计与可解释方法
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.inspection import permutation_importance
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

try:
    from scipy.stats import spearmanr
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

# ======== 配置：按需修改 ========
dataset = "cora_ml"
metric_name = "f1"          # 可改为 'roc_auc', 'pr_auc', 'mrr', 'hit100', 'hit50', 'hit20'
split_ratio_select = 0.9    # 选择一个存在于 CSV 的 split_ratio
csv_candidates = [
    f"./results/{dataset}_grid_result.csv",
    f"/mnt/data/{dataset}_grid_result.csv",
]

# ======== 读取与类型标准化 ========
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("找不到 grid_result.csv。请检查路径。")

df = pd.read_csv(csv_path)

PARAM_COLS = ['split_ratio', 'pos_ratio', 'c', 'h', 'gamma', 'alpha']
for col in ['split_ratio','pos_ratio','alpha']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in ['c','h','gamma']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

metric_cols = [c for c in df.columns if c not in PARAM_COLS]
if metric_name not in metric_cols:
    raise ValueError(f"指标 {metric_name} 不在文件中。可选：{metric_cols}")

df_r = df[df['split_ratio'] == split_ratio_select].copy()
if df_r.empty:
    raise ValueError(f"split_ratio={split_ratio_select} 无数据。候选：{sorted(df['split_ratio'].dropna().unique().tolist())}")

# ======== 定义性能：对每个 (pos_ratio, c, h, gamma) 取该指标在所有 alpha 下的最大值 ========
group_cols = ['pos_ratio', 'c', 'h', 'gamma']
perf_df = (
    df_r.groupby(group_cols, dropna=False)[metric_name]
        .max()
        .reset_index()
        .rename(columns={metric_name: 'performance'})
)

# ======== Top K 组合 ========
topk = perf_df.sort_values('performance', ascending=False)
os.makedirs("./analysis_out", exist_ok=True)
top_path = f"./analysis_out/{dataset}_{metric_name}_top_configs.csv"
perf_df.to_csv(f"./analysis_out/{dataset}_{metric_name}_perf_by_group.csv", index=False)
topk.head(200).to_csv(top_path, index=False)

# ======== 单参“主效应”统计：均值 ± 标准差 ========
def aggregate_effects(perf: pd.DataFrame, param: str) -> pd.DataFrame:
    agg = perf.groupby(param, dropna=False)['performance'].agg(['count', 'mean', 'std']).reset_index()
    agg = agg.rename(columns={'mean':'mean_perf', 'std':'std_perf'})
    return agg.sort_values(param)

agg_pos = aggregate_effects(perf_df, 'pos_ratio')
agg_c   = aggregate_effects(perf_df, 'c')
agg_h   = aggregate_effects(perf_df, 'h')
agg_g   = aggregate_effects(perf_df, 'gamma')

agg_path = f"./analysis_out/{dataset}_{metric_name}_aggregate_effects.xlsx"
with pd.ExcelWriter(agg_path, engine='xlsxwriter') as writer:
    perf_df.to_excel(writer, sheet_name='perf_by_group', index=False)
    topk.head(200).to_excel(writer, sheet_name='top200_configs', index=False)
    agg_pos.to_excel(writer, sheet_name='pos_ratio_effect', index=False)
    agg_c.to_excel(writer, sheet_name='c_effect', index=False)
    agg_h.to_excel(writer, sheet_name='h_effect', index=False)
    agg_g.to_excel(writer, sheet_name='gamma_effect', index=False)

# ======== 胜率（“条件最优频次”）：衡量离散超参的稳健性 ========
# 对于每个“其他超参”的组合，统计当前超参的哪个取值拿到最高 performance（获胜），计算各取值胜率。
def win_rate(perf: pd.DataFrame, target_param: str) -> pd.DataFrame:
    others = [p for p in group_cols if p != target_param]
    # 对每个 others 的组合，找 target_param 哪个值赢
    winners = (
        perf.sort_values(['performance'], ascending=False)
            .groupby(others, dropna=False)
            .head(1)[[target_param]]
    )
    rates = winners.value_counts().rename('wins').reset_index()
    total = winners.shape[0]
    rates['win_rate'] = rates['wins'] / total
    return rates.sort_values(['win_rate', target_param], ascending=[False, True]), total

wr_pos, n_pos = win_rate(perf_df, 'pos_ratio')
wr_c,   n_c   = win_rate(perf_df, 'c')
wr_h,   n_h   = win_rate(perf_df, 'h')
wr_g,   n_g   = win_rate(perf_df, 'gamma')

wr_path = f"./analysis_out/{dataset}_{metric_name}_win_rates.xlsx"
with pd.ExcelWriter(wr_path, engine='xlsxwriter') as writer:
    wr_pos.to_excel(writer, sheet_name='pos_ratio_win_rate', index=False)
    wr_c.to_excel(writer, sheet_name='c_win_rate', index=False)
    wr_h.to_excel(writer, sheet_name='h_win_rate', index=False)
    wr_g.to_excel(writer, sheet_name='gamma_win_rate', index=False)

# ======== 单调性：斯皮尔曼秩相关（如可用） ========
mono_rows = []
if SCIPY_OK:
    for p in group_cols:
        x = perf_df[p].astype(float).values
        y = perf_df['performance'].values
        r, pv = spearmanr(x, y)
        mono_rows.append({'param': p, 'spearman_r': r, 'p_value': pv})
mono_df = pd.DataFrame(mono_rows)
mono_path = f"./analysis_out/{dataset}_{metric_name}_monotonicity.csv"
mono_df.to_csv(mono_path, index=False)

# ======== 模型型重要性（如可用）：Random Forest + 置换重要性 ========
if SKLEARN_OK:
    X = perf_df[['pos_ratio','c','h','gamma']].astype(float).values
    y = perf_df['performance'].values
    rf = RandomForestRegressor(
        n_estimators=600, random_state=42, n_jobs=-1, max_depth=None
    )
    rf.fit(X, y)
    perm = permutation_importance(rf, X, y, n_repeats=20, random_state=42, n_jobs=-1)
    imp_df = pd.DataFrame({
        'feature': ['pos_ratio','c','h','gamma'],
        'rf_importance': rf.feature_importances_,
        'perm_importance_mean': perm.importances_mean,
        'perm_importance_std': perm.importances_std
    }).sort_values('perm_importance_mean', ascending=False)
    imp_path = f"./analysis_out/{dataset}_{metric_name}_importance.csv"
    imp_df.to_csv(imp_path, index=False)

print("完成。输出文件：")
print(" - Top 组合:", top_path)
print(" - 单参主效应(均值±std):", agg_path)
print(" - 胜率统计:", wr_path)
print(" - 单调性(Spearman):", mono_path)
if SKLEARN_OK:
    print(" - 重要性(RF & Permutation):", imp_path)


完成。输出文件：
 - Top 组合: ./analysis_out/cora_ml_f1_top_configs.csv
 - 单参主效应(均值±std): ./analysis_out/cora_ml_f1_aggregate_effects.xlsx
 - 胜率统计: ./analysis_out/cora_ml_f1_win_rates.xlsx
 - 单调性(Spearman): ./analysis_out/cora_ml_f1_monotonicity.csv
 - 重要性(RF & Permutation): ./analysis_out/cora_ml_f1_importance.csv
